# Homework 2 — Mini-proyecto avanzado de nowcasting

## From Deterministic Forecasts to Probabilistic Nowcasting: Skill, Uncertainty, and Failure Modes

**Duración sugerida:** 10 horas  
**Nivel:** intermedio-avanzado  
**Formato:** mini-reporte científico, no solo ejercicio de código.

En este proyecto vas a conducir una evaluación completa de nowcasting usando múltiples muestras, múltiples métricas y, si están disponibles, predicciones de EarthFormer y CasCast. La meta es diagnosticar calidad de pronóstico, incertidumbre, desplazamiento espacial, extremos y generalización regional.

## Pregunta científica principal

Elige **una pregunta principal** y **una secundaria**:

- **A — Degradación con horizonte:** ¿cómo cae la habilidad desde 5 hasta 60 minutos?
- **B — Detección de lluvia intensa:** ¿el modelo detecta precipitación intensa o solo lluvia ligera/moderada?
- **C — Generalización regional:** ¿Barranca y Guaviare tienen la misma habilidad?
- **D — Determinístico vs probabilístico:** ¿CasCast mejora estructura local y extremos frente a EarthFormer?
- **E — Incertidumbre y ensamble:** ¿múltiples muestras CasCast aportan incertidumbre útil?

Escribe aquí tu elección antes de empezar:

> Pregunta principal:  
> Pregunta secundaria:

## 0. Prerrequisitos

Antes de iniciar, desde `nowcasting_course_lab/` debes tener al menos datos y persistencia:

```bash
python scripts/download_assets.py
python scripts/make_persistence_predictions.py
```

Si quieres comparar modelos, primero genera predicciones:

```bash
python scripts/run_model_inference.py --stage earthformer --sample all --device auto
python scripts/run_model_inference.py --stage all --sample all --device auto --ddim-steps 20
python scripts/evaluate_predictions.py --sample all
```

En CPU, prueba primero:

```bash
python scripts/run_model_inference.py --stage all --smoke --device cpu --ddim-steps 2 --cpu-threads 8
```

In [ ]:
from pathlib import Path
import sys
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from course_utils.data import (
    SAMPLE_FILES,
    LEAD_MINUTES,
    get_paths,
    load_sample,
    split_sequence,
    make_persistence_prediction,
)
from course_utils.palette import apply_course_style, RAIN_LEVELS
from course_utils.plotting import plot_event_grid, plot_target_prediction_panel

apply_course_style()
paths = get_paths(PROJECT_ROOT)

THRESHOLDS = [0.5, 2.0, 5.0, 10.0]
POOL_SCALES = [1, 4, 8, 16]

print(f"Proyecto: {paths.root}")
print(f"Muestras: {paths.sample_dir}")
print(f"Niveles de lluvia para color: {RAIN_LEVELS}")

## 1. Auditoría del dataset

Inspecciona todas las muestras. Debes producir una tabla con:

- `filename`
- `region`
- `datetime`
- `H`, `W`
- `n_frames`
- `max_rain_rate`
- `mean_rain_rate`
- `rainy_pixel_fraction`, usando lluvia `>= 0.5 mm/h`
- `heavy_rain_pixel_fraction`, usando lluvia `>= 5 mm/h`
- `event_class`, elegida por ti

Clases sugeridas:

- `mostly dry`
- `stratiform/light rain`
- `localized convective core`
- `spatially widespread rain`
- `fast-moving system`

In [ ]:
def parse_sample_metadata(filename):
    """TODO: extrae region y fecha/hora desde el nombre del archivo.

    Ejemplo:
    barranca_seq_20240426_2000_patch_04_rain_rate.npy
    region = barranca
    datetime = 2024-04-26 20:00
    """
    # Pista: puedes usar split("_") o una expresión regular.
    raise NotImplementedError("Completa parse_sample_metadata")


def classify_event(sequence):
    """TODO: asigna una clase cualitativa al evento.

    Sugerencia simple:
    - si fracción lluviosa baja -> mostly dry
    - si max alto y área pesada pequeña -> localized convective core
    - si fracción lluviosa alta -> spatially widespread rain
    - si lluvia media/moderada sin máximos fuertes -> stratiform/light rain
    """
    raise NotImplementedError("Completa classify_event")


def build_dataset_audit(sample_files=SAMPLE_FILES):
    rows = []
    for filename in sample_files:
        sequence = load_sample(filename, paths)
        # TODO: usa parse_sample_metadata(filename)
        # TODO: calcula H, W, n_frames, max, mean, rainy fraction, heavy fraction
        # TODO: agrega event_class = classify_event(sequence)
        raise NotImplementedError("Completa build_dataset_audit")
    return pd.DataFrame(rows)

In [ ]:
# Cuando completes la auditoría:
# dataset_audit = build_dataset_audit(SAMPLE_FILES)
# dataset_audit

## 2. Descubrir predicciones disponibles

El proyecto debe funcionar con diferentes niveles:

- mínimo: persistencia;
- mejor: EarthFormer;
- más completo: EarthFormer + CasCast;
- avanzado: varias muestras CasCast como ensamble.

Esta celda te ayuda a detectar qué predicciones existen actualmente.

In [ ]:
def available_predictions_for_case(name):
    """Carga target y predicciones disponibles para un archivo."""
    sequence = load_sample(name, paths)
    inputs, target = split_sequence(sequence)
    predictions = {"persistence": make_persistence_prediction(inputs)}

    for model_name, folder in [
        ("earthformer", paths.earthformer_dir),
        ("cascast", paths.cascast_dir),
        ("model", paths.model_dir),
    ]:
        pred_path = folder / name
        if pred_path.exists():
            predictions[model_name] = np.nan_to_num(
                np.load(pred_path).astype(np.float32),
                nan=0.0,
                posinf=0.0,
                neginf=0.0,
            )
    return inputs, target, predictions

availability = []
for name in SAMPLE_FILES:
    _, _, preds = available_predictions_for_case(name)
    availability.append({"sample": name, "available_predictions": ", ".join(preds)})

pd.DataFrame(availability)

## 3. Visualización inicial: mismo caso, varios modelos

Escoge un caso y compara visualmente persistencia, EarthFormer y CasCast si existen.

In [ ]:
sample_name = SAMPLE_FILES[0]
inputs, target, predictions = available_predictions_for_case(sample_name)
print("Modelos disponibles:", list(predictions))

for model_name, pred in predictions.items():
    fig = plot_event_grid(inputs, target, pred, sample_name, model_name, lead_indices=(2, 5, 8, 11))
    plt.show()

## 4. Métricas continuas

Implementa métricas por lead time:

$$
MAE_k = \frac{1}{HW}\sum_{i,j}|\hat{Y}_{k,i,j}-Y_{k,i,j}|
$$

$$
RMSE_k = \sqrt{\frac{1}{HW}\sum_{i,j}(\hat{Y}_{k,i,j}-Y_{k,i,j})^2}
$$

$$
Bias_k = \frac{1}{HW}\sum_{i,j}(\hat{Y}_{k,i,j}-Y_{k,i,j})
$$

También calcula correlación de Pearson por lead time.

In [ ]:
def safe_pearson(x, y):
    """TODO: correlación segura entre dos campos 2D."""
    raise NotImplementedError("Completa safe_pearson")


def compute_continuous_metrics(pred, target):
    """TODO: devuelve DataFrame con lead_min, MAE, RMSE, Bias, Pearson_r."""
    rows = []
    for i, lead in enumerate(LEAD_MINUTES):
        # TODO: calcula error, MAE, RMSE, Bias, Pearson
        raise NotImplementedError("Completa compute_continuous_metrics")
    return pd.DataFrame(rows)

In [ ]:
# Prueba en un caso/modelo cuando completes la función:
# continuous_example = compute_continuous_metrics(predictions["persistence"], target)
# continuous_example

## 5. Métricas categóricas de eventos

Para umbrales `0.5`, `2`, `5`, `10 mm/h`, calcula:

- TP, FP, FN, TN
- CSI
- POD
- FAR
- Precision
- Recall
- F1

$$
CSI = \frac{TP}{TP+FP+FN}
$$

$$
POD = \frac{TP}{TP+FN}, \qquad FAR = \frac{FP}{TP+FP}
$$

In [ ]:
def safe_divide(numerator, denominator):
    """TODO: devuelve np.nan si denominator == 0."""
    raise NotImplementedError("Completa safe_divide")


def compute_binary_metrics(pred, target, threshold):
    """TODO: calcula conteos y métricas binarias para un umbral."""
    raise NotImplementedError("Completa compute_binary_metrics")


def compute_event_metrics(pred, target, thresholds=THRESHOLDS):
    """TODO: aplica compute_binary_metrics para cada lead time y umbral."""
    rows = []
    for threshold in thresholds:
        for i, lead in enumerate(LEAD_MINUTES):
            # TODO: metrics = compute_binary_metrics(pred[i], target[i], threshold)
            # TODO: append threshold, lead, metrics
            raise NotImplementedError("Completa compute_event_metrics")
    return pd.DataFrame(rows)

## 6. Métricas por escala espacial: POOL1, POOL4, POOL8, POOL16

La lluvia puede estar desplazada unos pixeles. Por eso evaluamos CSI después de max-pooling espacial:

- **POOL1:** exactitud pixel a pixel.
- **POOL4:** amenaza local.
- **POOL8:** vecindario más amplio.
- **POOL16:** amenaza regional.

La idea es aplicar max-pooling antes de umbralizar.

In [ ]:
def max_pool_2d(field, pool_size):
    """TODO: max-pooling 2D para H,W divisibles por pool_size.

    Pista: reshape a (H//p, p, W//p, p) y toma max en ejes (1, 3).
    """
    raise NotImplementedError("Completa max_pool_2d")


def max_pool_sequence(sequence, pool_size):
    """TODO: aplica max_pool_2d a una secuencia (T,H,W)."""
    raise NotImplementedError("Completa max_pool_sequence")


def compute_pooled_csi(pred, target, threshold=5.0, pool_scales=POOL_SCALES):
    """TODO: calcula CSI por lead time, threshold y escala de pooling."""
    rows = []
    for pool in pool_scales:
        # TODO: pool pred y target
        # TODO: para cada lead time calcula CSI usando compute_binary_metrics
        raise NotImplementedError("Completa compute_pooled_csi")
    return pd.DataFrame(rows)

## 7. Métricas basadas en objetos

Threshold a `5 mm/h`, identifica componentes conectados y evalúa:

- número de objetos;
- área del objeto más grande;
- distancia entre centroides del objeto principal predicho y observado;
- lluvia máxima dentro del objeto;
- IoU del objeto principal.

Esto ayuda a distinguir: “el modelo encontró la tormenta, pero desplazada” vs “el modelo no encontró la tormenta”.

In [ ]:
try:
    from scipy import ndimage
except ImportError as exc:
    raise ImportError("Instala scipy: conda install -c conda-forge scipy") from exc


def largest_object_properties(field, threshold=5.0):
    """TODO: propiedades del objeto de lluvia más grande en un frame 2D."""
    # Pista:
    # mask = field >= threshold
    # labels, n = ndimage.label(mask)
    # si n == 0: devuelve valores np.nan / 0
    # calcula área por etiqueta, elige la mayor, centroide con ndimage.center_of_mass
    raise NotImplementedError("Completa largest_object_properties")


def object_metrics_frame(pred_frame, target_frame, threshold=5.0):
    """TODO: compara objeto principal predicho vs observado."""
    # Debe devolver: n_pred, n_target, largest_area_pred, largest_area_target,
    # centroid_distance, max_rate_pred_object, max_rate_target_object, object_iou
    raise NotImplementedError("Completa object_metrics_frame")


def compute_object_metrics(pred, target, threshold=5.0):
    """TODO: aplica object_metrics_frame por lead time."""
    rows = []
    for i, lead in enumerate(LEAD_MINUTES):
        # TODO
        raise NotImplementedError("Completa compute_object_metrics")
    return pd.DataFrame(rows)

## 8. Análisis de extremos

Calcula por frame:

- máximo de lluvia;
- percentil 95;
- percentil 99;
- área `>= 5 mm/h`;
- área `>= 10 mm/h`.

Gráfico requerido: máximo observado vs máximo predicho.

In [ ]:
def extreme_stats_sequence(sequence):
    """TODO: métricas de extremos por lead time para una secuencia (T,H,W)."""
    rows = []
    for i, lead in enumerate(LEAD_MINUTES):
        frame = sequence[i]
        # TODO: max, p95, p99, area_ge_5, area_ge_10
        raise NotImplementedError("Completa extreme_stats_sequence")
    return pd.DataFrame(rows)


def compare_extremes(pred, target, model_name):
    """TODO: une extremos de predicción y target en una tabla."""
    raise NotImplementedError("Completa compare_extremes")

## 9. Baselines opcionales: suavizado y extrapolación

### Baseline de suavizado

Aplica Gaussian blur a una predicción y compara:

- ¿mejora RMSE?
- ¿empeora CSI a 5 o 10 mm/h?
- ¿baja la intensidad máxima?

Esto muestra por qué optimizar pérdidas pixel-wise puede producir campos suaves.

### Extrapolación lineal opcional

Si quieres ir más allá, estima un desplazamiento entre los dos últimos frames de entrada y desplaza el último frame hacia adelante. Puedes usar fase de correlación, búsqueda de shift simple o `scipy`.

In [ ]:
from scipy.ndimage import gaussian_filter, shift as scipy_shift


def smooth_prediction(pred, sigma=1.5):
    """TODO: aplica gaussian_filter por frame."""
    raise NotImplementedError("Completa smooth_prediction")


def estimate_simple_shift(frame_a, frame_b, max_shift=8):
    """OPCIONAL TODO: estima desplazamiento entero entre dos frames por búsqueda de correlación."""
    raise NotImplementedError("Opcional: completa estimate_simple_shift")


def linear_extrapolation_baseline(inputs, pred_frames=12):
    """OPCIONAL TODO: desplaza el último frame repetidamente usando el shift estimado."""
    raise NotImplementedError("Opcional: completa linear_extrapolation_baseline")

## 10. Pipeline final de evaluación

Integra todo para todos los archivos y todos los modelos disponibles. Tu tabla final debe tener al menos:

- `sample`
- `region`
- `model`
- `RMSE_mean`
- `Bias_mean`
- `Pearson_mean`
- `CSI_0.5_mean`
- `CSI_5_mean`
- `CSI_10_mean`
- `POOL1_CSI5_mean`
- `POOL4_CSI5_mean`
- `POOL16_CSI5_mean`
- `centroid_distance_mean`
- `target_max_mean`
- `pred_max_mean`

In [ ]:
def evaluate_all_models_advanced(sample_files=SAMPLE_FILES):
    """TODO: evaluación avanzada por archivo y modelo."""
    rows = []
    for filename in sample_files:
        inputs, target, preds = available_predictions_for_case(filename)
        # TODO: region desde filename
        for model_name, pred in preds.items():
            # TODO: continuous, event, pooled, object, extreme
            # TODO: resumir en una fila
            raise NotImplementedError("Completa evaluate_all_models_advanced")
    return pd.DataFrame(rows)

In [ ]:
# Cuando completes el pipeline:
# advanced_summary = evaluate_all_models_advanced(SAMPLE_FILES)
# advanced_summary

## 11. Visualizaciones requeridas

Incluye al menos:

1. RMSE vs lead time.
2. Bias vs lead time.
3. Correlación vs lead time.
4. CSI vs lead time por umbral.
5. FAR vs umbral.
6. POD vs umbral.
7. CSI por escala espacial: POOL1, POOL4, POOL8, POOL16.
8. Target max intensity vs predicted max intensity.
9. Un panel visual target/predicción para 2–3 casos.

In [ ]:
# TODO: crea aquí tus funciones de plotting del reporte.
# Pistas:
# - usa groupby model para comparar curvas
# - usa plt.plot para lead time
# - usa plt.scatter para target max vs pred max
# - usa plot_target_prediction_panel para paneles visuales

## 12. Ensamble e incertidumbre, si tienes múltiples CasCast samples

Si generaste varias muestras CasCast, analiza:

- ensemble mean;
- ensemble standard deviation;
- spread vs error absoluto;
- si zonas de alta incertidumbre coinciden con errores grandes.

Formato sugerido para archivos:

```text
outputs/predictions/cascast_ens/member_00/<sample>.npy
outputs/predictions/cascast_ens/member_01/<sample>.npy
...
```

In [ ]:
def load_cascast_ensemble(sample_name, ensemble_root=None):
    """OPCIONAL TODO: carga miembros CasCast si existen."""
    ensemble_root = ensemble_root or (paths.prediction_dir / "cascast_ens")
    # TODO: busca member_* / sample_name
    raise NotImplementedError("Opcional: completa load_cascast_ensemble")


def ensemble_uncertainty_metrics(ensemble_pred, target):
    """OPCIONAL TODO: spread, error del ensemble mean, correlación spread-error."""
    raise NotImplementedError("Opcional: completa ensemble_uncertainty_metrics")

## 13. Estructura del mini-reporte

Tu entrega debe parecer un reporte científico corto:

1. **Pregunta y motivación**: opción principal + secundaria.
2. **Datos**: auditoría, regiones, tipos de evento.
3. **Métodos**: modelos/baselines, métricas, umbrales, pooling, objetos.
4. **Resultados**: tablas y figuras principales.
5. **Discusión**: degradación con horizonte, extremos, desplazamiento, generalización regional.
6. **Limitaciones**: pocos casos, CPU/GPU, número de muestras CasCast, incertidumbre.
7. **Conclusión**: 3–5 hallazgos concretos.

## Rúbrica sugerida

| Componente | Puntos |
|---|---:|
| Auditoría y clasificación de eventos | 15 |
| Inferencia/baselines y organización reproducible | 15 |
| Métricas continuas y categóricas | 20 |
| Pooling espacial, objetos y extremos | 25 |
| Interpretación científica y figuras | 20 |
| Claridad/reproducibilidad | 5 |
| Total | 100 |